# NyayaRAG Colab Artifact Builder

Run this notebook on a Colab GPU. It only builds `nyayarag_artifacts.zip` in Google Drive. Download that zip and unzip it locally to run Streamlit inference.

In [ ]:
!nvidia-smi

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = os.environ["HOME"] + "/.local/bin:" + os.environ["PATH"]
!uv --version

In [ ]:
%cd /content
!rm -rf NyayaRAG
!git clone https://github.com/mi-bilal/NyayaRAG.git
%cd /content/NyayaRAG

## Configure HF Token

Option A: add `HF_TOKEN` in Colab secrets and run the next cell. Option B: replace `your_hf_token_here` manually in the fallback cell. Groq is not needed for artifact building.

In [ ]:
from google.colab import userdata

hf = userdata.get("HF_TOKEN") or ""
env = f"""HF_TOKEN={hf}
GROQ_MODEL=openai/gpt-oss-120b
EMBEDDING_MODEL=Qwen/Qwen3-Embedding-0.6B
EMBEDDING_DIM=1024
QDRANT_PATH=artifacts/qdrant
QDRANT_COLLECTION=nyayarag_statutes
BM25_PATH=artifacts/bm25/statutes_bm25.pkl
DATA_DIR=data
ARTIFACT_DIR=artifacts
TOP_K=5
CANDIDATE_POOL=80
RRF_K=60
MAX_CASE_TOKENS=3500
MAX_CONTEXT_TOKENS=5000
"""
open(".env", "w").write(env)
print("Wrote .env; HF token present:", bool(hf))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!uv sync --dev
!uv run python -c "import accelerate; print('accelerate', accelerate.__version__)"

## Small Test Build

Run this first. It builds a small zip to prove everything works. Uses a separate cache dir so it doesn't interfere with the full build.

In [ ]:
!rm -rf artifacts
!uv run python scripts/colab_full_build.py --batch-size 16 --upsert-batch-size 512 --embed-max-tokens 384 --embed-overlap-tokens 48 --log-every 1 --limit 50 --cache-dir /content/nyayarag_test_cache

In [ ]:
import os, json
cache_dir = '/content/drive/MyDrive/nyayarag_cache'
cp_path = os.path.join(cache_dir, 'checkpoint.json')
if os.path.exists(cp_path):
    cp = json.load(open(cp_path))
    print('Cache checkpoint found:')
    for k, v in cp.items():
        print(f'  {k}: {v}')
    embed_done = cp.get('completed_up_to', 0)
    total = cp.get('total_rows', 0)
    if embed_done and total:
        print(f'\nEmbedding progress: {embed_done:,}/{total:,} rows ({100*embed_done/total:.1f}%)')
else:
    print('No cache found — full build will start from scratch.')

## Full Build (with checkpoint caching)

Run this after the small test succeeds. The cache lives on Google Drive so if the runtime disconnects, re-running this cell picks up where it left off (corpus download, chunk extraction, and partially-encoded vectors are all cached).

In [ ]:
!rm -rf artifacts
!uv run python scripts/colab_full_build.py --batch-size 64 --upsert-batch-size 1024 --embed-max-tokens 512 --embed-overlap-tokens 64 --log-every 10 --cache-dir /content/drive/MyDrive/nyayarag_cache

In [ ]:
!ls -lh /content/drive/MyDrive/nyayarag_artifacts/

Download `/content/drive/MyDrive/nyayarag_artifacts/nyayarag_artifacts.zip`, unzip it into your local repo root, then run `uv run streamlit run app/streamlit_app.py`.